# Prayer Time Scheduler

The purpose of this notebook is to add to Google Calendar the iqamah time at the local masjid (for me: ISBCC).

## Constants

In [ ]:
# Website for Masjid timings https://timing.athanplus.com/masjid/widgets/monthly?masjid_id={MASJID_ID}
# Calendar ID found under GCalendar settings

MASJID_ID: str = "zVKp9PLP"
GMAIL_ADDRESS: str = "awab.masroor@gmail.com"
URL: str = f"https://timing.athanplus.com/masjid/widgets/monthly?masjid_id={MASJID_ID}"
CALNEDAR_ID: str = "9552e4095afc087ef09b180ecc2d5b7c510149687b090c454ce55c26050311a1@group.calendar.google.com"
MASJID: str = "ISBCC"

In [ ]:
PRAYER_TIMES: set = {
    "FAJR",
    "DHUHR",
    "ASR",
    "MAGHRIB",
    "ISHA",
}

### Install Dependencies

In [ ]:
%pip install -r requirements.txt

## Code

### Imports

In [ ]:
# Consolidated imports: standard lib, third-party, google APIs
import os
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Google API imports
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google_calendar_proxy import GoogleCalendarProxy

# Notes: keep `requirements.txt` updated rather than inline pip installs.

### HTTP Request for Masjid Timetable

In [ ]:
# get website
response = requests.get(URL)

if response.status_code == 200:
    print("Success")
else:
    print("Failure")

In [ ]:
soup = BeautifulSoup(response.text, 'html.parser')

# get the table
table = soup.find_all(id="time-table3")[0]

### Parse HTTP request

In [ ]:
# get the headers (class = subHeader)
headers = table.find_all(class_="subHeader")
headers = [header.text.strip() for header in headers]
headers

In [ ]:
# get the rows
rows = []
for tr in table.find_all('tr')[1:]:  # Skip the header row
    cols = tr.find_all('td')
    row = [col.get_text().strip() for col in cols]
    rows.append(row)

In [ ]:
df = pd.DataFrame(rows, columns=headers)

In [ ]:
# get the dates
dates = []
for index, row in df.iterrows():    
    # get date
    current_date = datetime.now()
    month = current_date.strftime("%B").upper()
    day = row[month]
    year = current_date.year
    # parse the number of the day
    day = int(day.split()[0].replace(',', ''))
    date = datetime.strptime(f"{year} {month} {day}", "%Y %B %d")
    
    dates.append(date)    

In [ ]:
new_df = pd.DataFrame(dates, columns=["Date"])

In [ ]:
for prayer in PRAYER_TIMES:
    new_df[f"{prayer} Adhan"] = df[prayer].apply(lambda x: x.split()[0])
    new_df[f"{prayer} Iqamah"] = df[prayer].apply(lambda x: x.split()[2])

# remove the old columns
new_df.head(10)

In [ ]:
date_df_data = []

# go through the rows and create events
for index, row in new_df.iterrows():
    new_row = {}
    new_row['Date'] = row['Date']

    for col in new_df.columns.drop('Date'):
        if ("DHUHR" in col or "ASR" in col or "MAGHRIB" in col or "ISHA" in col) and "11:" not in row[col]:
            dt = datetime.combine(row['Date'], datetime.strptime(f'{row[col]}', "%I:%M").time())
            dt = dt + timedelta(hours=12)
        else:
            dt = datetime.combine(row['Date'], datetime.strptime(row[col], "%H:%M").time())

        new_row[col] = dt

    date_df_data.append(new_row)

date_df = pd.DataFrame(date_df_data)

In [ ]:
date_df.head(3)

### Grab credentials to add events to calendar

In [ ]:
# grab google credentials
# (Google API imports moved to the consolidated imports cell above)

creds = None
SCOPES = ['https://www.googleapis.com/auth/calendar']

if os.path.exists('token.json'):
    if (datetime.now() - datetime.fromtimestamp(os.path.getmtime('token.json'))) > timedelta(hours=24):
        print("The creds file is over 24 hours old. Deleting creds")
        os.remove('token.json')
    else:
        creds = Credentials.from_authorized_user_file('token.json')

# If there are no (valid) credentials available, let the user log in.
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(
            'credentials.json', SCOPES)
        creds = flow.run_local_server(port=0)
    # Save the credentials for the next run
    with open('token.json', 'w') as token:
        token.write(creds.to_json())

## Create events

In [ ]:
# Define calendar service
service = build('calendar', 'v3', credentials=creds)

In [ ]:
proxy = GoogleCalendarProxy(service, CALNEDAR_ID)

tz = ZoneInfo('America/New_York')

# calendar events mapping
first_ms_month = datetime.now(timezone.utc).replace(day=1, hour=0, minute=0, second=0, microsecond=0)
events = proxy.list_events(time_min=first_ms_month.isoformat().replace('+00:00', 'Z'))  # RFC3339 UTC
calendar_map = proxy.events_map_by_start(events)

# build local_map: start_datetime -> [event_body, ...]
local_map = {}
for index, row in date_df.iterrows():
    for prayer in PRAYER_TIMES:
        start = row[prayer + ' Iqamah']
        # make sure start is timezone-aware to match calendar events
        if start.tzinfo is None:
            start = start.replace(tzinfo=tz)
        end = start + timedelta(minutes=10)
        summary = f"{prayer.title()} Salah"
        description = f"{prayer.title()} Salah at {MASJID}"
        reminders = [{'method': 'popup', 'minutes': 30}, {'method': 'popup', 'minutes': 15}]
        body = proxy.build_event_body(summary, 'https://maps.app.goo.gl/kC7nVr2kWGn4pNV96', start, end, description, reminders)
        local_map.setdefault(start, []).append(body)

# print local mapping summary
print('Local event start mapping (total starts={}):'.format(len(local_map)))
for k, v in sorted(local_map.items()):
    print(k, '->', [e['summary'] for e in v])

# iterate and insert missing events
for start in sorted(list(local_map.keys())):
    bodies = local_map.pop(start)
    for body in bodies:
        if not proxy.event_exists(body['summary'], start, calendar_map):
            print('Adding event:', body['summary'], 'at', start)
            proxy.insert_event(body)
        else:
            print('Already exists, skipping:', body['summary'], 'at', start)